In [16]:
import os
from dotenv import load_dotenv
load_dotenv()
#hf_token = os.getenv("HF_TOKEN")
#print(hf_token)
#print(os.getenv("HF_HOME"))

True

Completely Offline Deployment\
Once downloaded, you can run without internet:

In [3]:
from transformers import AutoModel
model = AutoModel.from_pretrained("openai/privacy-filter")

config.json: 0.00B [00:00, ?B/s]

e:\Work\GitHub\banban\architecture-as-code\notebooks\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\Work\GitHub\banban\architecture-as-code\notebooks\.hf-cache\hub\models--openai--privacy-filter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.80G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[transformers] OpenAIPrivacyFilterModel LOAD REPORT from: openai/privacy-filter
Key          | Status     |  | 
-------------+------------+--+-
score.weight | UNEXPECTED |  | 
score.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "openai/privacy-filter",
    use_fast=False
)

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

Check if model is cached

In [11]:
from huggingface_hub import scan_cache_dir

print(scan_cache_dir())

HFCacheInfo(size_on_disk=2826860945, repos=frozenset({CachedRepoInfo(repo_id='openai/privacy-filter', repo_type='model', repo_path=WindowsPath('E:/Work/GitHub/banban/architecture-as-code/notebooks/.hf-cache/hub/models--openai--privacy-filter'), size_on_disk=2826860945, nb_files=4, revisions=frozenset({CachedRevisionInfo(commit_hash='7ffa9a043d54d1be65afb281eddf0ffbe629385b', snapshot_path=WindowsPath('E:/Work/GitHub/banban/architecture-as-code/notebooks/.hf-cache/hub/models--openai--privacy-filter/snapshots/7ffa9a043d54d1be65afb281eddf0ffbe629385b'), size_on_disk=2826860945, files=frozenset({CachedFileInfo(file_name='tokenizer_config.json', file_path=WindowsPath('E:/Work/GitHub/banban/architecture-as-code/notebooks/.hf-cache/hub/models--openai--privacy-filter/snapshots/7ffa9a043d54d1be65afb281eddf0ffbe629385b/tokenizer_config.json'), blob_path=WindowsPath('E:/Work/GitHub/banban/architecture-as-code/notebooks/.hf-cache/hub/models--openai--privacy-filter/snapshots/7ffa9a043d54d1be65afb28

Load model from local cache (offline-safe)

In [12]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_PATH = "openai/privacy-filter"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    use_fast=False   # important fallback
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

pipe = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

Run inference

In [15]:
def redact(text, entities):
    for e in reversed(entities):
        text = text[:e["start"]] + "[REDACTED]" + text[e["end"]:]
    return text

text = "My name is John Doe and my email is john.doe@gmail.com"


entities = pipe(text)
print(redact(text, entities))

My name is[REDACTED][REDACTED] and my email is[REDACTED][REDACTED]
